In [30]:
using LowLevelFEM

In [31]:
openGeometry("mpc-1.geo")

#openPreProcessor()

In [32]:
mat1 = Material("body")
mat2 = Material("remote")

U = Field([mat1, mat2], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
#Φ = Field([mat1, mat2], type=:ScalarField, dim=2, fieldName=:φ, rhsName=:m)

Problem("mpc-1", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)

In [33]:
Ku = ∫(SymGrad(U) ⋅ [2 1 0; 1 2 0; 0 0 1] ⋅ SymGrad(U), Ω="body")
Ku[:,:]

10×10 SparseArrays.SparseMatrixCSC{Float64, Int64} with 62 stored entries:
  1.0           0.5          -0.5          …  -1.38778e-17   ⋅    ⋅ 
  0.5           1.0          -3.46945e-18     -0.5           ⋅    ⋅ 
 -0.5          -3.46945e-18   1.0              0.5           ⋅    ⋅ 
  1.04083e-17   5.55112e-17  -0.5             -0.5           ⋅    ⋅ 
 -0.5          -0.5           5.55112e-17       ⋅            ⋅    ⋅ 
 -0.5          -0.5           1.38778e-17  …   5.55112e-17   ⋅    ⋅ 
  5.55112e-17   1.38778e-17  -0.5             -0.5           ⋅    ⋅ 
 -1.38778e-17  -0.5           0.5              1.0           ⋅    ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅    ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅    ⋅ 

In [34]:
#Kφ = ∫(U ⋅ U * 0, Ω="body")
#Kφ[:,:]

In [35]:
mpc = MPC(master="remote", slave="right"; field=U, ux=true, uy=true)

#vagy egy mező esetén (nincs nyomaték)

mpc = MPC(master="remote", slave="right", uy=false)
bc2 = BoundaryCondition("remote", uy=0)

BoundaryCondition("remote", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0))

In [36]:
bc = BoundaryCondition("left", ux=0, uy=0)

fu = ∫(U ⋅ [1, 0], Γ="remote")
DoFs(fu);

In [37]:
u = solveField(Ku, fu, support=[bc, bc2], mpc=[mpc])

nodal VectorField
[0.0; 0.0; … ; 0.5999999999999999; 0.0;;]

In [38]:
showDoFResults(u)

0

In [39]:
openPostProcessor()